# Q1: Create a ML system to comprehend instructions such as "Play the highest-rated songs by the Beatles"
**Topic:** System-design | **Difficulty:** Hard | **Time:** 20 min
**Sheet:** Grind75ML

---

## Question
Create a machine learning system to comprehend instructions such as "Play the highest-rated songs by the Beatles."

## System Design

### High-Level Architecture
```
User Voice/Text → ASR (if voice) → NLU Pipeline → Query Builder → Music DB → Playback
```

### Component 1: Speech-to-Text (ASR)
- **Model**: Whisper (OpenAI) or Conformer-based ASR
- **Input**: Audio waveform → **Output**: Transcribed text
- **Considerations**: Noise robustness, streaming vs batch, accent handling

### Component 2: Natural Language Understanding (NLU)
The core ML component. Needs to extract:

**Intent Classification:**
- `play_music`, `pause_music`, `skip_song`, `create_playlist`, `search_music`
- Model: Fine-tuned BERT/DistilBERT classifier or LLM with function calling

**Named Entity Recognition (NER) / Slot Filling:**
| Slot | Example | Type |
|------|---------|------|
| artist | Beatles | ARTIST |
| sort_criteria | highest-rated | SORT |
| genre | rock | GENRE |
| playlist | favorites | PLAYLIST |
| count | top 5 | NUMBER |
| year | from the 90s | DATE_RANGE |

- Model: Token classification (BERT + CRF) or seq2seq slot filling

**Example parse:**
```
Input:  "Play the highest-rated songs by the Beatles"
Intent: play_music
Slots:  { artist: "Beatles", sort: "rating_desc" }
```

### Component 3: Dialog Manager
- Handles multi-turn conversations: "Play Beatles" → "Which album?" → "Abbey Road"
- Tracks context/state across turns
- Can be rule-based FSM or learned (GPT-based dialog)

### Component 4: Query Builder
Converts structured intent + slots into a database query:
```sql
SELECT * FROM songs 
WHERE artist = 'Beatles' 
ORDER BY rating DESC;
```

### Component 5: Music Database
- **Metadata**: song, artist, album, genre, year, rating, play_count
- **Search**: Elasticsearch for fuzzy matching ("Beetles" → "Beatles")
- **Recommendations**: Collaborative filtering for "similar to Beatles"

### ML Training Pipeline
1. **Data Collection**: User query logs + annotations
2. **Intent model**: Multi-class classification on labeled queries
3. **NER model**: Token-level BIO tagging on annotated queries
4. **Evaluation**: Intent accuracy, slot F1, end-to-end task completion rate

### Modern Approach: LLM with Function Calling
Instead of separate intent/NER models, use an LLM (GPT-4, Llama) with function calling:
```json
{
  "function": "play_music",
  "parameters": {
    "artist": "Beatles",
    "sort_by": "rating",
    "order": "desc"
  }
}
```
- Pros: Handles complex/ambiguous queries, easy to extend
- Cons: Latency, cost, less control

### System Diagram
```
┌──────────┐    ┌─────────┐    ┌──────────────┐    ┌──────────┐
│  User    │───→│  ASR    │───→│  NLU/LLM     │───→│  Query   │
│  Input   │    │(Whisper)│    │(Intent+Slots) │    │  Builder │
└──────────┘    └─────────┘    └──────────────┘    └────┬─────┘
                                                         │
┌──────────┐    ┌─────────────┐    ┌──────────┐        │
│ Playback │←───│  Ranking/   │←───│ Music DB │←───────┘
│ Service  │    │  RecSys     │    │(Elastic) │
└──────────┘    └─────────────┘    └──────────┘
```

In [ ]:
import numpy as np
from dataclasses import dataclass
from enum import Enum


class Intent(Enum):
    PLAY_MUSIC = 'play_music'
    SEARCH_MUSIC = 'search_music'
    PAUSE = 'pause'
    SKIP = 'skip'
    CREATE_PLAYLIST = 'create_playlist'


@dataclass
class ParsedQuery:
    intent: Intent
    artist: str = None
    genre: str = None
    sort_by: str = None
    limit: int = None


# Simplified rule-based NLU (in production, use BERT/LLM)
def parse_query(text: str) -> ParsedQuery:
    """Simple keyword-based NLU parser (demo only)."""
    text_lower = text.lower()
    
    # Intent detection
    if any(w in text_lower for w in ['play', 'listen']):
        intent = Intent.PLAY_MUSIC
    elif any(w in text_lower for w in ['search', 'find']):
        intent = Intent.SEARCH_MUSIC
    elif 'pause' in text_lower or 'stop' in text_lower:
        intent = Intent.PAUSE
    else:
        intent = Intent.SEARCH_MUSIC
    
    # Slot extraction (simplified)
    artist = None
    known_artists = ['beatles', 'queen', 'pink floyd', 'led zeppelin', 'taylor swift']
    for a in known_artists:
        if a in text_lower:
            artist = a.title()
    
    sort_by = None
    if any(w in text_lower for w in ['highest-rated', 'top rated', 'best']):
        sort_by = 'rating_desc'
    elif 'popular' in text_lower or 'most played' in text_lower:
        sort_by = 'play_count_desc'
    elif 'latest' in text_lower or 'newest' in text_lower:
        sort_by = 'year_desc'
    
    return ParsedQuery(intent=intent, artist=artist, sort_by=sort_by)


# --- Demo ---
queries = [
    "Play the highest-rated songs by the Beatles",
    "Find popular Queen tracks",
    "Listen to the latest Taylor Swift album",
    "Pause the music",
]

for q in queries:
    result = parse_query(q)
    print(f"Query:  {q}")
    print(f"Parsed: {result}\n")

## Key Takeaways
- Modern systems use **LLM function calling** instead of separate intent/NER models — simpler and more flexible
- **Entity resolution** is critical — fuzzy matching handles typos and variations
- Design for **multi-turn dialog** from the start — real users rarely give perfect single-turn commands
- Measure **end-to-end task completion rate**, not just NLU accuracy